# TryOn Studio ? resumable CatVTON worker

Run the **Resume runtime** cell below. It is intentionally safe to rerun:

- Google Drive retains the source tree, pip cache and Hugging Face model cache.
- The current VM skips dependencies that are already installed.
- An already healthy worker/tunnel is reused.
- A fresh VM rebuilds only ephemeral processes and produces a new free tunnel URL.

Before the first run, select **Runtime ? Change runtime type ? T4 GPU**. In Colab's Secrets panel, add `GITHUB_TOKEN` (read access to the private repository) and `TRYON_WORKER_TOKEN` (a random value of at least 24 characters), and enable notebook access for both.


In [ ]:
# One resumable cell: safe to run on the first launch and after every disconnect.
from google.colab import drive, userdata
from getpass import getpass
from pathlib import Path
import base64
import os
import subprocess

drive.mount("/content/drive")

PERSISTENT_ROOT = Path("/content/drive/MyDrive/TryOnStudio")
PROJECT_ROOT = PERSISTENT_ROOT / "source"
REPOSITORY = "https://github.com/CloudyPatil/ai-virtual-try-on-extension.git"
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

def read_secret(name, prompt):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    return value or getpass(prompt)

github_token = None
if not (PROJECT_ROOT / ".git").exists():
    github_token = read_secret("GITHUB_TOKEN", "GitHub token (input hidden): ")
    credential = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    subprocess.run(
        ["git", "-c", f"http.extraHeader=Authorization: Basic {credential}", "clone", REPOSITORY, str(PROJECT_ROOT)],
        check=True,
    )
else:
    print("Persistent project source found; skipping clone.")

worker_token = read_secret("TRYON_WORKER_TOKEN", "Worker token, at least 24 characters (input hidden): ")
os.environ["HF_HOME"] = str(PERSISTENT_ROOT / "huggingface")
os.environ["HF_HUB_CACHE"] = str(PERSISTENT_ROOT / "huggingface" / "hub")

subprocess.run(
    [
        "python",
        str(PROJECT_ROOT / "services" / "inference-worker" / "colab_runtime.py"),
        "--project-root",
        str(PROJECT_ROOT),
        "--persistent-root",
        str(PERSISTENT_ROOT),
        "--token",
        worker_token,
    ],
    check=True,
)


## After restart

Copy the printed `TRYON_PROVIDER_URL` into the local backend's `.env`. The worker token remains unchanged when stored in Colab Secrets. A free Quick Tunnel receives a new URL when its process or VM restarts, so only that one setting normally changes.


In [ ]:
# Optional diagnostics: run this only if the Resume cell reports a failure.
from pathlib import Path

log_root = Path("/content/tryon-studio/logs")
for name in ("worker.log", "tunnel.log"):
    path = log_root / name
    print(f"\n===== {name} =====")
    print(path.read_text(errors="replace")[-6000:] if path.exists() else "Log not created.")
